In [1]:
"""
Construction Project Data Acquisition, Simulation, and Cleaning Pipeline
Week 1 Data Science Internship Deliverable
"""

import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler


def generate_construction_dataset(n_projects: int = 500, random_seed: int = 42) -> pd.DataFrame:
    """Simulates realistic construction project performance metrics with noise and correlations."""
    np.random.seed(random_seed)

    project_types = ["Residential", "Commercial", "Infrastructure", "Industrial"]
    zones = ["Urban", "Suburban", "Rural"]

    # Base features
    planned_duration = np.random.randint(90, 1000, size=n_projects)
    planned_budget = np.random.exponential(scale=15_000_000, size=n_projects) + 500_000
    material_cost_ratio = np.random.uniform(0.30, 0.65, size=n_projects)
    subcontractor_reliability = np.clip(np.random.normal(0.82, 0.12, size=n_projects), 0.35, 1.0)
    weather_delay_days = np.random.poisson(lam=planned_duration / 50.0)
    safety_incidents = np.random.poisson(lam=0.8, size=n_projects)

    # Realistic delay and cost inflation dynamics
    # Delays driven by weather, low subcontractor reliability, and safety stop-work orders
    delay_factor = (
        (weather_delay_days * 1.5)
        + ((1.0 - subcontractor_reliability) * 120)
        + (safety_incidents * 14)
        + np.random.normal(0, 15, size=n_projects)
    )
    actual_duration = planned_duration + np.maximum(0, delay_factor.astype(int))

    # Actual cost driven by schedule creep, material ratio surges, and labor overruns
    cost_creep_pct = (
        ((actual_duration - planned_duration) / planned_duration) * 0.45
        + (material_cost_ratio * 0.15)
        + np.random.normal(0.02, 0.08, size=n_projects)
    )
    actual_cost = planned_budget * (1.0 + cost_creep_pct)

    labor_hours = (actual_duration * 8 * np.random.uniform(12, 60, size=n_projects)).round(1)

    df = pd.DataFrame(
        {
            "Project_ID": [f"PRJ-{10000 + i}" for i in range(n_projects)],
            "Project_Type": np.random.choice(project_types, size=n_projects, p=[0.35, 0.35, 0.20, 0.10]),
            "Location_Zone": np.random.choice(zones, size=n_projects, p=[0.50, 0.35, 0.15]),
            "Planned_Duration_Days": planned_duration,
            "Actual_Duration_Days": actual_duration,
            "Planned_Budget_USD": planned_budget.round(2),
            "Actual_Cost_USD": actual_cost.round(2),
            "Material_Cost_Ratio": material_cost_ratio.round(3),
            "Labor_Hours_Logged": labor_hours,
            "Weather_Delay_Days": weather_delay_days,
            "Safety_Incidents_Count": safety_incidents,
            "Subcontractor_Score": subcontractor_reliability.round(3),
        }
    )

    # Inject realistic field data anomalies (approx. 4% missing values in observational columns)
    mask_weather = np.random.rand(n_projects) < 0.04
    mask_score = np.random.rand(n_projects) < 0.05
    mask_labor = np.random.rand(n_projects) < 0.03

    df.loc[mask_weather, "Weather_Delay_Days"] = np.nan
    df.loc[mask_score, "Subcontractor_Score"] = np.nan
    df.loc[mask_labor, "Labor_Hours_Logged"] = np.nan

    return df


def clean_and_prepare_pipeline(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Executes imputation, EVM domain engineering, outlier clipping, and robust scaling."""
    df = raw_df.copy()

    # 1. Domain-specific Missing Value Imputation
    # Forward/mean imputation for isolated site logs; KNN for correlated physical metrics
    num_impute_cols = ["Weather_Delay_Days", "Subcontractor_Score", "Labor_Hours_Logged"]
    imputer = KNNImputer(n_neighbors=5)
    df[num_impute_cols] = imputer.fit_transform(df[num_impute_cols])
    df["Weather_Delay_Days"] = df["Weather_Delay_Days"].round().astype(int)

    # 2. Construction Domain Feature Engineering (EVM & Delays)
    df["Delay_Days"] = df["Actual_Duration_Days"] - df["Planned_Duration_Days"]
    df["Schedule_Overrun_Pct"] = (df["Delay_Days"] / df["Planned_Duration_Days"]) * 100
    df["Cost_Overrun_USD"] = df["Actual_Cost_USD"] - df["Planned_Budget_USD"]
    df["Cost_Overrun_Pct"] = (df["Cost_Overrun_USD"] / df["Planned_Budget_USD"]) * 100

    # Earned Value Management Metrics (assuming project completion state)
    # At completion, EV = Planned_Budget_USD
    df["Cost_Performance_Index_CPI"] = (df["Planned_Budget_USD"] / df["Actual_Cost_USD"]).round(4)
    df["Schedule_Performance_Index_SPI"] = (
        df["Planned_Duration_Days"] / df["Actual_Duration_Days"]
    ).round(4)

    # 3. Outlier Mitigation via IQR for financial variance
    q25 = df["Cost_Overrun_Pct"].quantile(0.25)
    q75 = df["Cost_Overrun_Pct"].quantile(0.75)
    iqr = q75 - q25
    lower_bound = q25 - (1.5 * iqr)
    upper_bound = q75 + (1.5 * iqr)
    df["Is_Cost_Anomaly"] = (df["Cost_Overrun_Pct"] < lower_bound) | (
        df["Cost_Overrun_Pct"] > upper_bound
    )

    # 4. Feature Scaling for Modeling
    scaler = RobustScaler()
    scaled_cols = ["Planned_Budget_USD", "Actual_Cost_USD", "Labor_Hours_Logged"]
    df[[f"{c}_scaled" for c in scaled_cols]] = scaler.fit_transform(df[scaled_cols])

    return df


if __name__ == "__main__":
    # Generate simulated construction portfolio
    raw_dataset = generate_construction_dataset(n_projects=500)
    raw_dataset.to_csv("construction_projects_raw.csv", index=False)
    print(f"Synthesized {len(raw_dataset)} construction records.")

    # Process through pipeline
    cleaned_dataset = clean_and_prepare_pipeline(raw_dataset)
    cleaned_dataset.to_csv("construction_projects_cleaned.csv", index=False)

    print("\n--- Pipeline Summary ---")
    print(
        cleaned_dataset[
            [
                "Project_ID",
                "Delay_Days",
                "Cost_Overrun_Pct",
                "Cost_Performance_Index_CPI",
                "Schedule_Performance_Index_SPI",
            ]
        ].head()
    )

Synthesized 500 construction records.

--- Pipeline Summary ---
  Project_ID  Delay_Days  Cost_Overrun_Pct  Cost_Performance_Index_CPI  \
0  PRJ-10000          47         19.391141                      0.8376   
1  PRJ-10001          65         26.431059                      0.7909   
2  PRJ-10002          75         19.517922                      0.8367   
3  PRJ-10003          52         11.978858                      0.8930   
4  PRJ-10004          10         13.899858                      0.8780   

   Schedule_Performance_Index_SPI  
0                          0.8033  
1                          0.8898  
2                          0.9268  
3                          0.8738  
4                          0.9515  


In [3]:
cleaned_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Project_ID                      500 non-null    str    
 1   Project_Type                    500 non-null    str    
 2   Location_Zone                   500 non-null    str    
 3   Planned_Duration_Days           500 non-null    int32  
 4   Actual_Duration_Days            500 non-null    int64  
 5   Planned_Budget_USD              500 non-null    float64
 6   Actual_Cost_USD                 500 non-null    float64
 7   Material_Cost_Ratio             500 non-null    float64
 8   Labor_Hours_Logged              500 non-null    float64
 9   Weather_Delay_Days              500 non-null    int64  
 10  Safety_Incidents_Count          500 non-null    int32  
 11  Subcontractor_Score             500 non-null    float64
 12  Delay_Days                      500 non-null   